# Sod Shock Tube (1D)

This notebook runs the classical 1D Sod shock tube Riemann problem with the initial states below.

Side | $p$ | $\rho$ | $v$
---|---|---|---
Left | 1.0 | 1.0 | 0.0
Right | 0.1795 | 0.25 | 0.0

The expected solution contains the standard wave pattern: a left-going rarefaction, a contact discontinuity, and a right-going shock.

Unlike `sod_1d.py` (a thin `caseMain()` wrapper meant to just be run), this notebook is meant to be **edited while it runs**: the initial-condition generation below calls the same case code (`warpSPH.cases.sod.sodCase`/`buildSod1D`) the script does, and the step loop stays unrolled in a cell instead of being hidden inside `warpSPH.runner.run()` -- that's the hook point for prototyping new physics or, later, a gradient step for the backprop-over-trajectory demo. Plotting is the one place this notebook does *not* reuse `sodCase.setupPlot`/`updatePlot`: that path goes through `warpSPH.runner.display.openWindow`/`pumpEvents`, which -- for reasons not fully understood, isolated by testing both side by side -- does not live-update inside a Jupyter cell in this environment even though it calls the exact same underlying `plotSod`/`plotSod_` functions. This notebook calls `plotSod`/`plotSod_` directly instead (`plt.subplots()`, `ax.clear()`, `fig.canvas.draw()`/`flush_events()`), matching the pattern every pre-`Case` notebook (e.g. `02-Linear_Wave.ipynb`) already used, which is confirmed to live-update correctly.

Export uses the trajectory scheme (`storeMode='trajectory'`, one growing `trajectory.h5`) -- see `sod_resume.ipynb`/`sod_resume.py` to resume from it.

Precision note: switching between single and double precision is controlled in the import/configuration cell below. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/01-Sod_Shock_Tube.gif)


In [ ]:
%matplotlib widget
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.sod import sodCase, states
from warpSPH.caseUtils import plotSod, plotSod_
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import torch
from tqdm.autonotebook import tqdm


In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on `sod_1d.py` (or any
# other CaseSpec-driven case script), made explicit and editable here.
# `sodCase.defaults`/`sodCase.params` are the same values the CLI script
# starts from -- anything not overridden below just keeps its case default.
spec = CaseSpec(caseName=sodCase.name, scheme=sodCase.scheme, params=dict(sodCase.params)) \
    .merged(**sodCase.defaults)

spec = spec.merged(
    # --- discretisation --------------------------------------------------
    nx=800,                    # particle count on the left side (right side is nx // samplingRatio)
    dim=1,
    L=2.0,

    # --- time stepping -----------------------------------------------------
    tLimit=0.15,
    dt=1e-3, adaptiveDt=True, cflFactor=0.3,

    # --- scheme --------------------------------------------------------
    kernel='B7',

    # --- output ------------------------------------------------------------
    plot=True, show=True, plotInterval=10,
    store=True,
    storeMode='trajectory',    # one growing trajectory.h5 (see sod_resume.ipynb)
    exportInterval=0.005,      # simulated-time interval between stored frames

    # --- Sod's own knobs (left/right Riemann states, gamma, ...) -----------
    params=dict(
        gamma=5 / 3,
        left_rho=1.0, left_pressure=1.0, left_velocity=0.0,
        right_rho=0.25, right_pressure=0.1795, right_velocity=0.0,
        smoothIC=True, samplingRatio=4,
    ),
)
spec


In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`sodCase.buildSystem` -> `buildSod1D`), not re-derived here.
ctx = buildContext(sodCase, spec)
sodCase.configureScheme(ctx)
system = sodCase.buildSystem(ctx)
runningState = system.initializeNewState()


In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct plotSod/plotSod_ + plt.subplots(), not sodCase.setupPlot -- see the
# intro cell for why.
fig = axis = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    left, right = states(ctx)
    fig, axis = plotSod(runningState.state, ctx.config, ctx.schemeConfig, ctx.config.domain,
                        ctx.param('gamma'), left, right,
                        plotReference=True, plotLabels=False, scatter=False, t_=runningState.t)
    fig.savefig(os.path.join(ctx.imagePath, 'frame_00000.png'))

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = sodCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=sodCase.extraFields)


In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation, an extra diagnostic, or (later) a gradient step for the
# backprop-over-trajectory demo can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt))

trajectory = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point -------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -----------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = sodCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if fig is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        for ax in axis.flatten():
            ax.clear()
        plotSod_(fig, axis, runningState.state, ctx.config, ctx.schemeConfig, ctx.config.domain,
                ctx.param('gamma'), left, right,
                plotReference=True, plotLabels=False, scatter=True, t_=runningState.t)
        fig.canvas.draw()
        fig.canvas.flush_events()
        fig.savefig(os.path.join(ctx.imagePath, f'frame_{i:05d}.png'))

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=sodCase.extraFields)


In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)
